<h1><center> Model evaluation results I</center></h1>

<h4><center> Author: Anisbel León Marcos${^1}$ </center></h4>
<h6><center> ${^1}$Institute for Tropospheric Research (TROPOS)
</center></h6>
<h5><center> Contact information: </center></h5>
<h5><center> heinold@tropos.de (Bernd Heinold) </center></h5>
<br/>


>Plotting the interpolated biomolecule concentration in comparison to observation.

>The data to execute this notebook are in file "model_seawater.pkl"


#### Import packges

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import math
import matplotlib as mpl
import seaborn as sns
import matplotlib.ticker as ticker
from sklearn.metrics import mean_squared_error
from scipy.stats import linregress


#### Set data directory path
#### Reading data

In [ ]:
data_dir = './'
df_all_groups = pd.read_pickle(data_dir+"model_seawater.pkl") 

#### Define functions to plot data 

In [ ]:
## Statistical indicators
def equat_stat(model,observ):
    """Function to compute statistics"""
# Root Mean Squared Error    
    model,observ = np.array(model), np.array(observ)
    rmse = np.sqrt(mean_squared_error(model,observ))

# Correlation coefficient (R)
    res_lin_reg = linregress(observ, model)
    pearsons_coeff = res_lin_reg.rvalue
    pval_corr = res_lin_reg.pvalue
    
# Mean Bias and Normalised Mean Bias (NMB)
    mean_bias = np.nanmean(np.subtract(model, observ))
    nmb = np.nansum(np.subtract(model, observ)) / np.nansum(observ)


# Normalised Mean Standard Deviation (NMSD)
    std_obs = np.std(observ)
    std_mod = np.std(model)
    nmsd = (std_mod-std_obs)/std_obs   
    
    print('No. observations:', len(observ), '\n',
    	'RMSE:', rmse, '\n',
         'Pearson:', pearsons_coeff, '\n',
         'pval:', pval_corr,'\n',
         'MB:', mean_bias,'\n',
         'NMB:', nmb,'\n')
    
    return [rmse, pearsons_coeff, pval_corr, mean_bias, nmb]


In [ ]:
def plot_text(dict_macrom,ax, c_na, ID,l0,l1,h1,mol_name, stat_name, loc_factor):
    """
    Creates the box plot figure with all biomolecules groups separated by dashed lines
    """
    mod_data = dict_macrom[dict_macrom['']=='Model']
    obs_data = dict_macrom[dict_macrom['']=='Observation']
    stat_all = equat_stat(mod_data[c_na], obs_data[c_na])

    box2 = f'NMB={np.round(stat_all[-1],2)} '
    box1 = mol_name
    ax.text(l0,50,box1,fontsize = '14', weight='bold',bbox={'facecolor': 'white', 'alpha': 0.5, 'pad': 10})
    ax.text(l1,h1,box2,fontsize = '10')
    
    for idx, s in enumerate(stat_name):
        n = str(len(obs_data[obs_data['Measurements']==s]))

        ax.text(idx+loc_factor, 0.35, f'n= {n}',fontsize = '10')




def box_plot_vert(ax, dict_df,ID,lim):
    """
    Function to create box plot and calculate statistics indices
    :var dict_df: dataframe with model interpolated values and seawater samples for all biomolecules and stations
    :param ID: biomolecules ID (pol, pro, lip)
    :param lim: y-axis upper limits
    :return: None
    """
    c_na = 'Concentration in the ocean ($mmol\ C\ m^{-3}$)'

    states_palette = sns.color_palette("YlGnBu", n_colors=2)
    bx = sns.boxplot(ax = ax,
                     data=dict_df, x="Measurements",
                     y=c_na, hue="", palette=states_palette,
                     flierprops={
                                'marker': 'd',             # Use diamond marker
                                'markersize': 5,           # Set marker size
                                'markerfacecolor': '#404040'},
                     width=.7)

    
    # calculate statistics
    stations = {'pol  DCCHO [µMC]': ['NAO', 'WAP', 'CVAO  ', 'PUR12 ', 'PUR17'],
           'pro  DCAA [µMC]': [ 'CVAO', 'PUR12', 'SB', 'NWAO', 'SATL', 'WMED'],
           'lip  PG': ['CVAO ', 'AS']}
    
    pol_df = dict_df[dict_df['Macromolecules']=='pol  DCCHO [µMC]']
    pro_df = dict_df[dict_df['Macromolecules']=='pro  DCAA [µMC]']
    lip_df = dict_df[dict_df['Macromolecules']=='lip  PG']    
       
    plot_text(pol_df, ax, c_na, ID[0],0.1,3,50,'PCHO$_{sw}$|DCCHO$_{sw}$', stations['pol  DCCHO [µMC]'], 0)
    plot_text(pro_df, ax, c_na, ID[1],5,8,50,'DCAA$_{sw}$|DCAA$_{sw}$', stations['pro  DCAA [µMC]'], 5)
    plot_text(lip_df, ax, c_na, ID[2],11,10.8,20,'PL$_{sw}$|PG$_{sw}$', stations['lip  PG'], 11)

    # Customizing axes 
    ax.tick_params(axis = 'both',labelsize = '14')
    ax.yaxis.get_label().set_fontsize(14)
    ax.set_xlabel('',fontsize = 14)
    ax.set_yscale('log')
    ax.grid(linestyle='--', linewidth=0.4)
    ax.set_ylim(lim)

    #dotted lines to separate groups
    ax.axvline(4.5, color=".3", dashes=(2, 2))
    ax.axvline(10.5, color=".3", dashes=(2, 2))

    ax.legend(loc="lower left",fontsize = '14') 


#### Plotting data 

In [ ]:
# Create new plot
fig, ax= plt.subplots(figsize=(15, 8))
box_plot_vert(ax, df_all_groups,
              ['pol','pro','lip'],
              [1e-2,1e2])
plt.savefig(f'All_groups_box_plot.png',dpi = 300, bbox_inches="tight")
plt.close()
